# mine-tuning-model

## 필수 라이브러리 설치

In [1]:
!pip uninstall -y transformers trl -q
!pip install transformers==4.51.3 trl==0.17.0 -q
!pip install datasets peft accelerate bitsandbytes -q
!pip install torchao --upgrade -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip freeze > /content/drive/MyDrive/Colab\ Notebooks/Github/mine-tuning-model/requirements-ft.txt

# 데이터 로드 및 Instruction 형식 변환

In [4]:
from datasets import load_dataset

# 데이터 로드
ds = load_dataset("ddorin/minecraft-question-answer-260k")
print(ds)

# Instruction 형식으로 변환
def format_instruction(example):
    return {
        "text": f"""<|im_start|>system
You are a helpful Minecraft expert assistant.
<|im_end|>
<|im_start|>user
{example['question']}
<|im_end|>
<|im_start|>assistant
{example['answer']}
<|im_end|>"""
    }

train_data = ds["train"].map(format_instruction, remove_columns=["question", "answer", "source"])
print(f" 데이터 변환 완료: {len(train_data)}개")
print("\n샘플 확인:")
print(train_data[0]["text"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'source'],
        num_rows: 268239
    })
})
 데이터 변환 완료: 268239개

샘플 확인:
<|im_start|>system
You are a helpful Minecraft expert assistant.
<|im_end|>
<|im_start|>user
What types of leaves can be found naturally in jungle bushes in Minecraft?
<|im_end|>
<|im_start|>assistant
In Minecraft, jungle bushes can generate oak leaves in the Java Edition and jungle leaves in the Bedrock Edition.
<|im_end|>


# 모델, 토크나이저 로드

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id  = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
print(f"모델 로드 완료! 디바이스: {model.device}")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

모델 로드 완료! 디바이스: cuda:0


# LoRa 어댑터 설정

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


# 학습 실행(SFTTrainer)

In [ ]:
from trl import SFTTrainer, SFTConfig
from peft import PeftModel

# ============================================================
# 매 세션마다 이 부분만 수정!
# ============================================================
CHUNK_NUM   = 1        # 1회차: 1  /  2회차: 2
CHUNK_START = 0        # 1회차: 0  /  2회차: 130000
CHUNK_END   = 130000   # 1회차: 130000  /  2회차: 268239
DRIVE_DIR   = "/content/drive/MyDrive/Colab Notebooks/Github/mine-tuning-model"
# ============================================================

# 2회차: 이전 청크 모델 불러오기
if CHUNK_NUM > 1:
    prev_dir = f"{DRIVE_DIR}/chunk{CHUNK_NUM - 1}"
    print(f"이전 모델 로드 중: {prev_dir}")
    model = PeftModel.from_pretrained(model, prev_dir)
    model = model.merge_and_unload()
    model = get_peft_model(model, lora_config)
    print("✅ 이전 모델 병합 완료!")

# 청크 데이터 선택
train_chunk = train_data.select(range(CHUNK_START, CHUNK_END))
print(f"📦 {CHUNK_NUM}회차 학습 데이터: {len(train_chunk)}건 ({CHUNK_START}~{CHUNK_END})")

# 학습 설정
training_args = SFTConfig(
    output_dir=f"./minecraft-finetune-chunk{CHUNK_NUM}",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_chunk,
    args=training_args,
    processing_class=tokenizer,
)

print(f"🚀 {CHUNK_NUM}회차 학습 시작!")
trainer.train()

# Drive에 저장
save_dir = f"{DRIVE_DIR}/chunk{CHUNK_NUM}"
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ {CHUNK_NUM}회차 저장 완료! → {save_dir}")

📦 1회차 학습 데이터: 130000건 (0~130000)


Converting train dataset to ChatML:   0%|          | 0/130000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/130000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/130000 [00:00<?, ? examples/s]

In [32]:
import transformers
import trl

print(transformers.__version__)
print(trl.__version__)

5.0.0
1.4.0
